|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The roofline<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: prefill and decode are two different machines<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

The last notebook put a matmul on the roofline. Now put a real model on it.

One model does two jobs. Prefill reads a whole prompt at once. Decode makes
one token at a time. You will find them on opposite sides of the ridge. Almost
everything else in this course follows from that.

In [ ]:
### run this cell: the model, and what it weighs

model = AutoModelForCausalLM.from_pretrained(
          'Qwen/Qwen3-0.6B', dtype=torch.bfloat16).cuda().eval()

weight_bytes = sum(parameter.numel()*parameter.element_size() for parameter in model.parameters())
bandwidth = cudalib.peak_bandwidth(fresh=True)

print(f'weights:   {weight_bytes/1e9:.2f} GB')
print(f'bandwidth: {bandwidth:.0f} GB/s')
print(f'so one read of the weights costs at least {1000*weight_bytes/1e9/bandwidth:.2f} ms')

# Exercise 1: how fast is prefill

Time a forward pass over a prompt of length L. Use no cache. Then report the
cost per token.

In [ ]:
@torch.inference_mode()
def prefill_ms(num_tokens):
  token_ids = torch.randint(0, 1000, (1,num_tokens), device='cuda')
  return cudalib.bench_ms(lambda: model(token_ids, use_cache=False),
                          iters=10, warmup=3, best_of=2)

print(f"{'prompt':>7} {'ms':>9} {'ms/token':>10}")
for num_tokens in [128,256,512,1024,2048]:
  prefill_time = prefill_ms(num_tokens)
  print(f'{num_tokens:>7} {prefill_time:>9.2f} {prefill_time/num_tokens:>10.4f}')

# Exercise 2: how fast is decode

Now time one extra token. The context is already in the cache. A server does
this for every token after the first.

In [ ]:
import copy

@torch.inference_mode()
def decode_ms(num_tokens):
  token_ids = torch.randint(0, 1000, (1,num_tokens), device='cuda')
  cache = model(token_ids, use_cache=True).past_key_values
  next_token = torch.randint(0, 1000, (1,1), device='cuda')
  return cudalib.bench_ms(
      lambda: model(next_token, past_key_values=copy.copy(cache), use_cache=True),
      iters=10, warmup=3, best_of=2)

print(f"{'context':>8} {'ms per token':>13}")
for num_tokens in [128,512,2048]:
  print(f'{num_tokens:>8} {decode_ms(num_tokens):>13.2f}')

# Exercise 3: against the floor

A decode step must read every weight. That is a hard floor in milliseconds.
Compute the floor. Compare it with your measurement. Then turn the difference
into an achieved-bandwidth number.

In [ ]:
prefill_ms_per_token = prefill_ms(2048)/2048
decode_ms_per_token = decode_ms(2048)

floor_ms = 1000 * weight_bytes/1e9 / bandwidth
achieved = (weight_bytes/1e9) / (decode_ms_per_token/1000)

print(f'prefill: {prefill_ms_per_token:8.4f} ms/token')
print(f'decode:  {decode_ms_per_token:8.4f} ms/token   ({decode_ms_per_token/prefill_ms_per_token:.0f}x more, for the same model)')
print(f'\nthe floor for a decode step is {floor_ms:.2f} ms (one read of the weights)')
print(f'you measured                   {decode_ms_per_token:.2f} ms')
print(f'achieved bandwidth             {achieved:.0f} GB/s = {100*achieved/bandwidth:.0f}% of peak')

### Three numbers, and what each one tells you

**Decode costs hundreds of times more per token than prefill.** The weights
are the same. The arithmetic per weight is the same. Only one thing differs.

How many tokens share one read of the model? Prefill shares the read across
the whole prompt. Decode has one token. These are not two speeds of one
machine. They are two machines.

**Decode does not reach its own floor.** One read of the weights costs a few
milliseconds, and you measured several times that. The achieved bandwidth
therefore falls well below peak.

On a 0.6B model the GPU finishes each layer before Python asks for the next
one. You timed the asking. That gap does not describe the hardware. It is
stage 12, and CUDA graphs close it.

**The floor itself is real.** Batch 1 cannot beat it, and no change to your
code helps. Only one route passes the floor. One read of the weights must
serve more than one token. That is stages 04 and 05.